# Fine-tune Llama 3.2 for Urdu Drama Scripts (Kaggle, T4/P100)

QLoRA fine-tuning for Urdu drama-script generation. Pipeline: pull data -> build dataset -> stress test -> train -> evaluate -> upload to Hugging Face.

**Setup required (Kaggle Secrets - never hardcoded):**
- `HF_TOKEN` - Hugging Face token (gated model access + upload)
- `HF_USERNAME` - your Hugging Face username
- `HF_REPO` - target repo name (default: `urdu-drama-3.2-qLoRA`)

**Data:** enable the Kaggle dataset containing the `data/` folder from this repo, or let this notebook clone the repo directly. Path is auto-detected below.

In [ ]:
# Cell 1: Config and environment
import os
from pathlib import Path

CONFIG = {
    'model_name': 'meta-llama/Llama-3.2-3B-Instruct',
    'epochs': 2,
    'batch_size': 2,
    'gradient_accumulation': 8,
    'learning_rate': 2e-4,
    'max_seq_length': 2048,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'seed': 42,
}

HF_TOKEN = os.environ.get('HF_TOKEN')
HF_USERNAME = os.environ.get('HF_USERNAME', 'your-hf-username')
HF_REPO = os.environ.get('HF_REPO', 'urdu-drama-3.2-qLoRA')

print(f"Model: {CONFIG['model_name']}")
print(f"HF repo: {HF_USERNAME}/{HF_REPO}")
print(f"HF token configured: {bool(HF_TOKEN)}")
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable GPU in Kaggle settings'}")

In [ ]:
# Cell 2: Install what Kaggle does not preinstall (trl, bitsandbytes, metrics).
# Kaggle ships modern transformers 5.x / tokenizers 0.2x - do NOT downgrade; src/train.py adapts to both APIs.
%pip install -q --only-binary=:all: trl bitsandbytes evaluate rouge_score sacrebleu
import transformers, trl, tokenizers
print(f'transformers {transformers.__version__} | trl {trl.__version__} | tokenizers {tokenizers.__version__}')

In [ ]:
# Cell 3: Locate data (auto-detect Kaggle dataset input, else clone/pull repo)
import subprocess
import shutil

CANDIDATES = [
    '/kaggle/input/urdu-drama-data/data/seed',
    '/kaggle/input/llama-32-urdu/data/seed',
    '/kaggle/input/urdu-drama-data/seed',
]
SEED_DIR = next((c for c in CANDIDATES if Path(c).exists()), None)

if SEED_DIR is None:
    REPO = '/kaggle/working/repo'
    if (Path(REPO) / '.git').exists():
        print('Repo exists. Pulling latest...')
        r = subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], capture_output=True, text=True)
        print(r.stdout, r.stderr)
        if r.returncode != 0:
            print('Pull failed, re-cloning...')
            shutil.rmtree(REPO, ignore_errors=True)
            subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/danielosinor123-gif/llama-3.2-urdu.git', REPO], check=True)
    else:
        print('Cloning the repo...')
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/danielosinor123-gif/llama-3.2-urdu.git', REPO], check=True)
    os.chdir(REPO)
    SEED_DIR = 'data/seed'
else:
    os.makedirs('/kaggle/working/repo/data', exist_ok=True)
    os.chdir('/kaggle/working/repo')
    shutil.copytree(SEED_DIR, 'data/seed', dirs_exist_ok=True)
    SEED_DIR = 'data/seed'

print(f"Seed dir: {SEED_DIR}")
episodes = list(Path(SEED_DIR).glob('*.jsonl'))
print(f"Seed files: {[p.name for p in episodes]}")

In [ ]:
# Cell 4: Build dataset (filters + dedup + leakage-free splits)
!python src/data/build_dataset.py --seed_dir {SEED_DIR} --output_dir data/final

# Stress test MUST pass before training
!python src/stress_test.py --data_dir data/final --seed_dir {SEED_DIR}

import json
report = json.load(open('data/final/build_report.json'))
print('Split sizes:', report['split_sizes'])
print('Per source:', report['per_source'])

In [ ]:
# Cell 5: Train with QLoRA
HF_TOKEN = os.environ.get('HF_TOKEN') or HF_TOKEN  # re-read env (manual setup cell may have set it)
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN)

OUTPUT_DIR = 'urdu-drama-3.2-qLoRA'
c = CONFIG
train_cmd = (
    f'python src/train.py '
    f'--train_data data/final/train.jsonl '
    f'--eval_data data/final/val.jsonl '
    f'--output_dir {OUTPUT_DIR} '
    f"--model_name {c['model_name']} "
    f"--epochs {c['epochs']} "
    f"--batch_size {c['batch_size']} "
    f"--gradient_accumulation_steps {c['gradient_accumulation']} "
    f"--learning_rate {c['learning_rate']} "
    f"--max_seq_length {c['max_seq_length']} "
    f"--lora_r {c['lora_r']} "
    f"--lora_alpha {c['lora_alpha']} "
    f"--seed {c['seed']}"
)
print(train_cmd)
!{train_cmd}

In [ ]:
# Cell 6: Evaluate fine-tuned vs baseline (same prompts, greedy decoding)
eval_cmd = (
    f'python src/evaluate.py '
    f'--model_dir {OUTPUT_DIR} '
    f'--eval_data data/final/test.jsonl '
    f"--baseline_model {c['model_name']} "
    f'--output reports/eval.json'
)
!{eval_cmd}

import json
ev = json.load(open('reports/eval.json'))
print(f"{'Metric':<25}{'Baseline':<12}{'Fine-tuned':<12}")
for key in ['rougeL', 'bleu', 'chrf', 'script_structure_score']:
    b = ev.get('baseline', {}).get(key, '-')
    f = ev.get('fine_tuned', {}).get(key, '-')
    print(f"{key:<25}{str(b):<12}{str(f):<12}")

In [ ]:
# Cell 7: Upload adapter to Hugging Face (adapter only - small, not the base model)
from huggingface_hub import HfApi, login

def upload_to_hf(model_dir: str):
    if not HF_TOKEN:
        raise ValueError('HF_TOKEN missing. Add it via Kaggle Secrets.')
    login(token=HF_TOKEN)
    api = HfApi()
    repo_id = f'{HF_USERNAME}/{HF_REPO}'
    api.create_repo(repo_id=repo_id, exist_ok=True)
    api.upload_folder(folder_path=model_dir, repo_id=repo_id)
    print(f'Uploaded: https://huggingface.co/{repo_id}')

upload_to_hf(OUTPUT_DIR)

## After training

1. Find the notebook artifacts in the right panel (Output) or use the HF upload from Cell 7
2. Load the adapter with `peft.PeftModel.from_pretrained` or merge locally
3. Deploy: `ollama create urdu-drama -f ollama/Modelfile`
4. Fill in the benchmark table in `docs/BENCHMARKS.md` with the Cell 6 results

**Kaggle notes:**
- T4 (16GB) works for 3B QLoRA with batch_size 2 + grad accumulation 8
- If OOM: reduce `max_seq_length` to 1024 or batch_size to 1
- P100 lacks bfloat16 support - prefer T4 or A100 for this script (it uses bf16)